In [10]:
import pandas as pd
import numpy as np
from math import log

In [11]:
df = pd.read_csv("Loan_data.csv")

In [12]:
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [13]:
x = df['default'].to_list()
y = df['fico_score'].to_list()
n = len(x)

In [14]:
default = np.zeros(851, dtype=int)
total   = np.zeros(851, dtype=int)

idx = df["fico_score"].astype(int).to_numpy() - 300
np.add.at(default, idx, df["default"].to_numpy())
np.add.at(total,   idx, 1)

default[:551] = np.cumsum(default[:551])
total[:551]   = np.cumsum(total[:551])

In [15]:
def log_likelihood(n, k):
    p = k/n
    if (p==0 or p==1):
        return 0
    return k*np.log(p)+ (n-k)*np.log(1-p)

In [ ]:
r = 5
NEG = -10**18

dpv = np.full((r+1, 551), NEG, dtype=float)  
dpp = np.zeros((r+1, 551), dtype=int)        

dpv[0, :] = 0
dpv[1, :] = [log_likelihood(total[j], default[j]) for j in range(551)]

for i in range(2, r+1):
    for j in range(551):
        best, best_k = NEG, 0
        for k in range(j):
            if total[j] == total[k]:
                continue
            s = dpv[i-1, k] + log_likelihood(total[j]-total[k], default[j]-default[k])
            if s > best:
                best, best_k = s, k
        dpv[i, j], dpp[i, j] = best, best_k

print(round(float(dpv[r, 550]), 4))

k, cuts = 550, []
while r >= 0:
    cuts.append(k + 300)
    k = int(dpp[r, k])
    r -= 1
print(cuts)

C:\Users\samid\AppData\Local\Temp\ipykernel_5036\4170054945.py:2: RuntimeWarning: invalid value encountered in scalar divide
  p = k/n


-4255.3774
[850, 696, 640, 580, 520, 300]
